# DICE — Notebook 1.3 : incertitude et Monte-Carlo — Version étudiante

**Date de la séance :** mercredi 9 septembre 2026, 10:45–12:45  
**Parcours cible :** 1 h 30. Les cellules fournies traitent la mécanique Python ; votre travail porte sur la transposition, la comparaison et l'interprétation économique et climatique.


## 0) Préparation et importations


In [ ]:
# Ce notebook n'installe aucun paquet. L'environnement du cours doit être créé
# au préalable depuis le fichier requirements.txt, selon les instructions du README.
import importlib.util

required_modules = ["matplotlib", "numba", "numpy", "pandas", "scipy", "tqdm"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Paquets manquants : " + ", ".join(missing)
        + ". Installez l'environnement depuis requirements.txt avant le TP."
    )
print("Environnement Python prêt.")


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from DICE import (
    Params,
    init_states,
    update_path,
    mat_to_df,
    obj_fun,
    run_optimal_policy,
)

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 120
RNG = np.random.default_rng(42)

def clone_params(base, **overrides):
    candidate = Params()
    for name, value in vars(base).items():
        setattr(candidate, name, value)
    for name, value in overrides.items():
        setattr(candidate, name, float(value))
    return candidate

def simulate(base, **overrides):
    candidate = clone_params(base, **overrides)
    path = init_states(candidate)
    path[1:, candidate.i_s] = 0.20
    path[1:, candidate.i_mu] = np.linspace(0.03, 0.60, candidate.nT - 1)
    path = update_path(path, range(1, candidate.nT), candidate)
    return path, candidate

def damage_fraction(path, par):
    lagged_temperature = np.r_[path[0, par.i_T_AT], path[:-1, par.i_T_AT]]
    damages = par.a2 * lagged_temperature ** par.a3
    if par.a4 != 0:
        damages += np.where(lagged_temperature > par.a6,
                            par.a4 * lagged_temperature ** par.a5, 0.0)
    return damages

def quantile_bands(array):
    return np.quantile(np.asarray(array), [0.05, 0.50, 0.95], axis=0)


## Q1) Simuler la référence — exemple résolu

On simule DICE avec l'étalonnage de référence et des contrôles exogènes : taux d'épargne constant $s_t=0{,}20$ et taux de réduction $\mu_t$ passant de 0,03 à 0,60 d'ici 2060.

La cellule suivante constitue le **scénario de référence entièrement résolu**. Vérifiez les unités des axes et décrivez en deux phrases la dynamique de la température et de la production.


In [ ]:
p = Params()
baseline, _ = simulate(p)
années = baseline[:, p.i_time]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(années, baseline[:, p.i_T_AT], color="firebrick", lw=2)
axes[0].set(title="Température atmosphérique", xlabel="Année", ylabel="°C au-dessus du niveau préindustriel")
axes[1].plot(années, baseline[:, p.i_Y], color="navy", lw=2)
axes[1].set(title="Production mondiale", xlabel="Année", ylabel="Milliers de milliards USD 2010")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## Q2) Incertitude sur la sensibilité climatique `T2XCO2`

Nordhaus (2018) retient une distribution lognormale ajustée aux estimations d’Olson et al. (2012). Les paramètres sont $\mu=1{,}107$ et $\sigma=0{,}264$ ; la distribution de référence a une moyenne de 3,13 °C, une médiane de 3,03 °C et un écart-type de 0,843 °C.


### Q2-A) Tirer 1 000 valeurs de `T2XCO2` — exemple résolu

La sensibilité climatique à l'équilibre (ECS) est le réchauffement à long terme associé à un doublement du CO₂. La cellule fournit les tirages et leur histogramme.

**À faire :** relevez les quantiles à 5 %, 50 % et 95 %, puis expliquez pourquoi la distribution n'est pas symétrique.


In [ ]:
N = 1000
sigma_ecs = math.log(1.25) / norm.ppf(0.95)
ecs_draws = np.exp(math.log(p.T2XCO2) + sigma_ecs * RNG.standard_normal(N))
ecs_quantiles = np.quantile(ecs_draws, [0.05, 0.50, 0.95])

plt.hist(ecs_draws, bins=35, color="steelblue", edgecolor="white")
for q, label in zip(ecs_quantiles, ["q05", "médiane", "q95"]):
    plt.axvline(q, ls="--", lw=1.5, label=f"{label} = {q:.2f} °C")
plt.xlabel("Sensibilité climatique T2XCO2 (°C)")
plt.ylabel("Nombre de tirages")
plt.title("Distribution simulée de la sensibilité climatique")
plt.legend()
plt.tight_layout()
plt.show()

print(dict(zip(["q05", "q50", "q95"], np.round(ecs_quantiles, 3))))


### Q2-B) Propager l'incertitude dans DICE — boucle fournie

La boucle ci-dessous applique exactement les mêmes contrôles à chaque valeur de `T2XCO2`. Elle stocke la température, la production et la fraction de dommages.

**À vérifier :** les trois tableaux doivent avoir la forme `(1000, nombre_de_périodes)`.


In [ ]:
ecs_temperature = []
ecs_output = []
ecs_damages = []

for ecs in ecs_draws:
    path, par = simulate(p, T2XCO2=ecs)
    ecs_temperature.append(path[:, par.i_T_AT])
    ecs_output.append(path[:, par.i_Y])
    ecs_damages.append(damage_fraction(path, par))

ecs_temperature = np.asarray(ecs_temperature)
ecs_output = np.asarray(ecs_output)
ecs_damages = np.asarray(ecs_damages)

print("température :", ecs_temperature.shape)
print("production  :", ecs_output.shape)
print("dommages    :", ecs_damages.shape)
assert ecs_temperature.shape == (N, p.nT)


### Q2-C) Construire une bande d'incertitude — température résolue

L'exemple trace la bande 5–95 % et la médiane de température.

**À faire :** transposez la même méthode à la production `ecs_output`, puis commentez l'incertitude en 2100. Un scénario est ici une simulation conditionnelle, pas une prévision probabiliste du futur.


In [ ]:
q_ecs_T = quantile_bands(ecs_temperature)
i2100 = np.argmin(np.abs(années - 2100))

plt.fill_between(années, q_ecs_T[0], q_ecs_T[2], color="firebrick", alpha=0.20, label="Intervalle 5–95 %")
plt.plot(années, q_ecs_T[1], color="firebrick", lw=2, label="Médiane")
plt.plot(années, baseline[:, p.i_T_AT], color="black", ls="--", lw=1.2, label="Référence")
plt.xlabel("Année")
plt.ylabel("Température atmosphérique (°C)")
plt.title("Incertitude climatique transmise à la température")
plt.legend()
plt.tight_layout()
plt.show()

print("Température en 2100 [q05, q50, q95] :", np.round(q_ecs_T[:, i2100], 3), "°C")

# À vous : calculez q_ecs_Y = quantile_bands(ecs_output), puis reproduisez le graphique.


> **Votre réponse (3–4 phrases).** Comment la dissymétrie des tirages de sensibilité climatique se retrouve-t-elle dans la bande de température ? Pourquoi l'effet sur la production est-il indirect ?


## Q3) Incertitude sur le paramètre de dommages `a2`

Nordhaus (2018) souligne la forte dispersion des estimations de dommages. Nous représentons cette incertitude par une distribution positive dont l’écart-type correspond approximativement à 0,118 % de production par °C².


### Q3-A) Tirer 1 000 valeurs de `a2` — exemple résolu

Le coefficient `a2` transforme la température en fraction de production perdue dans la fonction de dommages de DICE. La distribution positive est fournie.

**À faire :** relevez les trois quantiles et rappelez l'unité économique de la sortie `damage_fraction`.


In [ ]:
sigma_a2 = math.log(2.0) / norm.ppf(0.95)
a2_draws = np.exp(math.log(p.a2) + sigma_a2 * RNG.standard_normal(N))
a2_quantiles = np.quantile(a2_draws, [0.05, 0.50, 0.95])

plt.hist(a2_draws, bins=35, color="darkorange", edgecolor="white")
for q, label in zip(a2_quantiles, ["q05", "médiane", "q95"]):
    plt.axvline(q, ls="--", lw=1.5, label=f"{label} = {q:.4f}")
plt.xlabel("Coefficient de dommages a2")
plt.ylabel("Nombre de tirages")
plt.title("Distribution simulée du coefficient de dommages")
plt.legend()
plt.tight_layout()
plt.show()

print(dict(zip(["q05", "q50", "q95"], np.round(a2_quantiles, 6))))


### Q3-B) Simuler conjointement sensibilité climatique et dommages — boucle fournie

On apparie ici le tirage $i$ de sensibilité climatique avec le tirage $i$ du coefficient de dommages. Cette hypothèse d'indépendance est une simplification.

**À faire :** exécutez la cellule, vérifiez les dimensions et expliquez quelle dépendance plausible entre climat et dommages pourrait manquer.


In [ ]:
joint_temperature = []
joint_output = []
joint_damages = []

for ecs, a2 in zip(ecs_draws, a2_draws):
    path, par = simulate(p, T2XCO2=ecs, a2=a2)
    joint_temperature.append(path[:, par.i_T_AT])
    joint_output.append(path[:, par.i_Y])
    joint_damages.append(damage_fraction(path, par))

joint_temperature = np.asarray(joint_temperature)
joint_output = np.asarray(joint_output)
joint_damages = np.asarray(joint_damages)

assert joint_temperature.shape == joint_output.shape == joint_damages.shape == (N, p.nT)
print("Tableaux conjoints :", joint_temperature.shape)


### Q3-C) Tracer les intervalles conjoints — graphique fourni

Les trois bandes sont tracées ensemble pour relier le mécanisme : **sensibilité climatique → température → dommages → production**.

**À faire :** relevez les valeurs en 2100 et identifiez la variable dont l'intervalle relatif est le plus large.


In [ ]:
q_joint_T = quantile_bands(joint_temperature)
q_joint_Y = quantile_bands(joint_output)
q_joint_D = quantile_bands(joint_damages)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
series = [
    (q_joint_T, "Température", "°C", "firebrick"),
    (q_joint_Y, "Production", "Milliers de milliards USD 2010", "navy"),
    (100 * q_joint_D, "Dommages", "% de la production", "darkorange"),
]
for ax, (bands, title, unit, color) in zip(axes, series):
    ax.fill_between(années, bands[0], bands[2], color=color, alpha=0.20)
    ax.plot(années, bands[1], color=color, lw=2)
    ax.set(title=title, xlabel="Année", ylabel=unit)
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("2100 — température (°C) :", np.round(q_joint_T[:, i2100], 3))
print("2100 — production       :", np.round(q_joint_Y[:, i2100], 3))
print("2100 — dommages (%)     :", np.round(100 * q_joint_D[:, i2100], 3))


> **Votre réponse (4–5 phrases).** Décrivez la chaîne causale entre les deux paramètres incertains et les trois sorties. Distinguez l'incertitude physique (`T2XCO2`) de l'incertitude économique (`a2`).


### Q3-D) Comparer les sources d'incertitude — calcul guidé

Le tableau ci-dessous compare les largeurs $q_{95}-q_{05}$ en 2100. Pour la température, l'ajout d'une incertitude sur `a2` ne devrait pas élargir directement la bande : dans cette simulation à contrôles exogènes, les dommages ne rétroagissent pas sur le climat.

**À faire :** interprétez les trois lignes et expliquez pourquoi la conclusion diffère selon la variable.


In [ ]:
q_ecs_Y = quantile_bands(ecs_output)
q_ecs_D = quantile_bands(ecs_damages)

widths_2100 = pd.DataFrame(
    {
        "ECS seule": [
            q_ecs_T[2, i2100] - q_ecs_T[0, i2100],
            q_ecs_Y[2, i2100] - q_ecs_Y[0, i2100],
            100 * (q_ecs_D[2, i2100] - q_ecs_D[0, i2100]),
        ],
        "ECS + a2": [
            q_joint_T[2, i2100] - q_joint_T[0, i2100],
            q_joint_Y[2, i2100] - q_joint_Y[0, i2100],
            100 * (q_joint_D[2, i2100] - q_joint_D[0, i2100]),
        ],
    },
    index=["Température (°C)", "Production", "Dommages (% production)"],
)
widths_2100.round(3)


> **Votre réponse (3–4 phrases).** Quelle source d'incertitude domine pour chaque sortie ? Pourquoi `a2` ne modifie-t-il pas directement la température dans cette expérience ?


## Q4) Incertitude sur la décarbonation `deltasig`

Nordhaus (2018) estime une incertitude annuelle sur la tendance de l’intensité carbone. Une régression sur 1960–2015 conduit à une erreur de prévision importante en 2100 ; cette estimation peut encore être trop faible si la série possède une racine unitaire.


### Q4-A) Tirer 1 000 valeurs de `deltasig` — exemple résolu

`deltasig` modifie la vitesse de décarbonation autonome de l'économie. On utilise ici une loi semi-normale positive à titre de stress test.

**À faire :** exécutez la cellule et décrivez ce que représente la queue droite de la distribution.


In [ ]:
deltasig_draws = np.abs(RNG.normal(loc=0.0, scale=0.02, size=N))
deltasig_quantiles = np.quantile(deltasig_draws, [0.05, 0.50, 0.95])

plt.hist(deltasig_draws, bins=35, color="seagreen", edgecolor="white")
for q, label in zip(deltasig_quantiles, ["q05", "médiane", "q95"]):
    plt.axvline(q, ls="--", lw=1.5, label=f"{label} = {q:.3f}")
plt.xlabel("Paramètre deltasig")
plt.ylabel("Nombre de tirages")
plt.title("Incertitude sur la décarbonation autonome")
plt.legend()
plt.tight_layout()
plt.show()

print(dict(zip(["q05", "q50", "q95"], np.round(deltasig_quantiles, 4))))


### Q4-B) Simuler les trois incertitudes — boucle fournie

La boucle ajoute `deltasig` aux deux incertitudes précédentes. Elle est fournie car l'objectif n'est pas de recopier une troisième fois la même mécanique Monte-Carlo.

**À faire :** vérifiez les dimensions, puis formulez le sens attendu de l'effet d'un `deltasig` élevé sur les émissions et la température avant d'afficher les résultats.


In [ ]:
all_temperature = []
all_output = []
all_damages = []

for ecs, a2, deltasig in zip(ecs_draws, a2_draws, deltasig_draws):
    path, par = simulate(p, T2XCO2=ecs, a2=a2, deltasig=deltasig)
    all_temperature.append(path[:, par.i_T_AT])
    all_output.append(path[:, par.i_Y])
    all_damages.append(damage_fraction(path, par))

all_temperature = np.asarray(all_temperature)
all_output = np.asarray(all_output)
all_damages = np.asarray(all_damages)

assert all_temperature.shape == all_output.shape == all_damages.shape == (N, p.nT)
print("Tableaux avec trois incertitudes :", all_temperature.shape)


### Q4-C) Comparer les intervalles — température résolue

L'exemple compare la température avec deux puis trois sources d'incertitude.

**À faire :** reproduisez la comparaison pour la production ou les dommages, puis concluez sur la source d'incertitude qui élargit le plus les résultats en 2100.


In [ ]:
q_all_T = quantile_bands(all_temperature)
q_all_Y = quantile_bands(all_output)
q_all_D = quantile_bands(all_damages)

plt.fill_between(années, q_joint_T[0], q_joint_T[2], color="darkorange", alpha=0.20, label="ECS + dommages")
plt.fill_between(années, q_all_T[0], q_all_T[2], color="seagreen", alpha=0.20, label="ECS + dommages + décarbonation")
plt.plot(années, q_all_T[1], color="seagreen", lw=2, label="Médiane, trois incertitudes")
plt.xlabel("Année")
plt.ylabel("Température atmosphérique (°C)")
plt.title("Effet de l'incertitude de décarbonation")
plt.legend()
plt.tight_layout()
plt.show()

print("2100 — température, trois incertitudes :", np.round(q_all_T[:, i2100], 3), "°C")

# À vous : comparez q_joint_Y à q_all_Y, ou q_joint_D à q_all_D.


In [ ]:
# Synthèse attendue (5 lignes maximum) :
# 1. Quelle incertitude est physique, économique ou liée à la transition ?
# 2. Quelle sortie est la plus sensible à chacune ?
# 3. Pourquoi ces bandes ne sont-elles pas des prévisions probabilistes ?
